In [1]:
import numpy as np
import pywt
from scipy.stats import skew, kurtosis
import pandas as pd


In [2]:
def extraer_features(latidos, picos_R, fs=500):
    features = []

    # 1. TEMPORALES / HRV
    intervalos_RR = np.diff(picos_R) / fs

    features.append(np.mean(intervalos_RR))                                    # media RR
    features.append(np.std(intervalos_RR))                                     # SDNN
    features.append(np.min(intervalos_RR))                                     # min RR
    features.append(np.max(intervalos_RR))                                     # max RR
    features.append(np.max(intervalos_RR) - np.min(intervalos_RR))            # rango RR
    features.append(60 / np.mean(intervalos_RR))                               # FC en bpm
    rmssd = np.sqrt(np.mean(np.diff(intervalos_RR) ** 2)) if len(intervalos_RR) > 1 else 0.0
    features.append(rmssd)                                                      # RMSSD

    # 2. MORFOLÓGICAS
    amp_R    = []
    amp_T    = []
    area_QRS = []
    dur_QRS  = []
    amp_Q    = []
    st_nivel = []

    for latido in latidos:
        pico_R_idx = 100  # pre = 0.2s * 500Hz = 100 muestras

        # Amplitud pico R
        amp_R.append(latido[pico_R_idx])

        # Amplitud onda T: máximo entre muestras 150-250 (zona post-QRS)
        amp_T.append(np.max(latido[150:250]))

        # Área bajo el QRS: zona 75-125
        zona_QRS = latido[75:125]
        area_QRS.append(np.trapz(np.abs(zona_QRS)))

        # Duración QRS: muestras por encima del 50% del pico R
        umbral_QRS = 0.5 * latido[pico_R_idx]
        dur_QRS.append(np.sum(zona_QRS > umbral_QRS))

        # Amplitud onda Q: mínimo en zona pre-R (75-100)
        # La onda Q patológica es negativa y clave para MI
        amp_Q.append(np.min(latido[75:100]))

        # Nivel segmento ST: media de la zona 125-155 (post-QRS, pre-onda T)
        # Elevación o depresión indica STTC o MI
        st_nivel.append(np.mean(latido[125:155]))

    for stat_list in [amp_R, amp_T, area_QRS, dur_QRS, amp_Q, st_nivel]:
        features.append(np.mean(stat_list))
        features.append(np.std(stat_list))

    #3. ESTADÍSTICAS
    for fn in [np.mean, np.std, skew, kurtosis, np.max, np.min]:
        vals = [fn(l) for l in latidos]
        features.append(np.mean(vals))
        features.append(np.std(vals))

    # 4. FRECUENCIALES 
    energias_d4 = []
    energias_d5 = []
    energias_d6 = []

    for latido in latidos:
        coeffs = pywt.wavedec(latido, 'db6', level=8)
        # coeffs[0]=a8, [1]=d8, [2]=d7, [3]=d6, [4]=d5, [5]=d4
        energias_d4.append(np.sum(coeffs[5] ** 2))  # banda QRS alta (~20-40 Hz)
        energias_d5.append(np.sum(coeffs[4] ** 2))  # banda QRS baja (~10-20 Hz)
        energias_d6.append(np.sum(coeffs[3] ** 2))  # banda P y T (~5-10 Hz)

    for energias in [energias_d4, energias_d5, energias_d6]:
        features.append(np.mean(energias))
        features.append(np.std(energias))

    # Ratio LF/HF: balance entre bandas de baja y alta frecuencia
    ratio_LF_HF = np.mean(energias_d6) / (np.mean(energias_d4) + 1e-8)
    features.append(ratio_LF_HF)

    return np.array(features)

In [3]:
Y                 = np.load('Y.npy')
picos_R_pacientes = np.load('picos_R.npy',           allow_pickle=True)
latidos_pacientes = np.load('latidos.npy',            allow_pickle=True)
pacientes_validos = np.load('pacientes_validos.npy',  allow_pickle=True)

df_pacientes = pd.read_csv("C:\\Users\\Ana\\Documents\\4º GITT+BA (2025-2026)\\TFG\\TFG\\ptbxl_database.csv")

print(f"Pacientes: {len(pacientes_validos)}")

Pacientes: 16984


In [4]:
X_feat = []

for i, record in enumerate(pacientes_validos):
    latidos  = latidos_pacientes[i].astype(np.float64)
    picos_R  = picos_R_pacientes[i]
    features = extraer_features(latidos, picos_R, fs=500)

    fila = df_pacientes.loc[
        df_pacientes['filename_hr'].str.contains(record, case=False, na=False)
    ]
    edad = fila['age'].values[0]
    sexo = 1.0 if fila['sex'].values[0] == 1 else 0.0
    vector_completo = np.append(features, [edad, sexo])
    X_feat.append(vector_completo)

X_feat = np.array(X_feat)

print("Shape X_feat:", X_feat.shape)  # debería ser (N, 40)
print("Shape Y:     ", Y.shape)

np.save('X_feat.npy', X_feat)
print("Guardado correctamente")

C:\Users\Ana\AppData\Local\Temp\ipykernel_19024\831460112.py:35: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  area_QRS.append(np.trapz(np.abs(zona_QRS)))
c:\Users\Ana\AppData\Local\Programs\Python\Python312\Lib\site-packages\pywt\_multilevel.py:43: UserWarning: Level value of 8 is too high: all coefficients will experience boundary effects.
  warnings.warn(


Shape X_feat: (16984, 40)
Shape Y:      (16984, 5)
Guardado correctamente
